# 02 — Sanity Check Model

Notebook ini memverifikasi:
- Model MAE-ViT backbone bisa di-load
- Forward pass dummy berjalan tanpa error
- Output shape sesuai harapan
- LoRA bisa di-attach dengan benar
- Jumlah parameter trainable sesuai

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
from src.utils.config import load_config
from src.utils.seed import set_seed

set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

## 1. Load Backbone

In [ ]:
from src.models.mae_vit import load_mae_vit_backbone, freeze_backbone, get_backbone_info

model_config = load_config('../configs/model_mae_vit.yaml')

backbone = load_mae_vit_backbone(
    model_name=model_config['backbone']['model_name'],
    pretrained=True,
    device=device
)

info = get_backbone_info(backbone)
print(f'\nBackbone Info:')
for k, v in info.items():
    print(f'  {k}: {v}')

## 2. Forward Pass Dummy

In [ ]:
# Dummy input: batch of 4 images, 3 channels, 224x224
dummy_input = torch.randn(4, 3, 224, 224).to(device)

with torch.no_grad():
    output = backbone(pixel_values=dummy_input, output_hidden_states=True)

print(f'Last hidden state shape: {output.last_hidden_state.shape}')
print(f'CLS token shape: {output.last_hidden_state[:, 0, :].shape}')
print(f'Num hidden states: {len(output.hidden_states)}')

# Expected: (4, 197, 768) — 196 patches + 1 CLS token
assert output.last_hidden_state.shape == (4, 197, 768), 'Shape mismatch!'
print('\n✓ Forward pass successful!')

## 3. Classifier Head

In [ ]:
from src.models.classifier_head import CassavaClassifier, build_classifier_head

head_config = model_config['classifier'].copy()
head_config['hidden_size'] = 768
head = build_classifier_head(head_config).to(device)

model = CassavaClassifier(backbone, head).to(device)

with torch.no_grad():
    result = model(pixel_values=dummy_input)

print(f'Logits shape: {result["logits"].shape}')
print(f'CLS embedding shape: {result["cls_embedding"].shape}')

assert result['logits'].shape == (4, 5), 'Logits shape mismatch!'
print('\n✓ Classifier head works!')

## 4. LoRA Attachment

In [ ]:
from src.models.lora_layers import attach_lora_peft, extract_lora_weights_peft
from src.models.mae_vit import load_mae_vit_backbone, freeze_backbone

# Fresh backbone for LoRA test
backbone_lora = load_mae_vit_backbone(device=device)
freeze_backbone(backbone_lora)

# Attach LoRA
backbone_lora = attach_lora_peft(
    backbone_lora, rank=8, alpha=16,
    target_modules=['query', 'value'],
    lora_dropout=0.05
)

# Forward pass
with torch.no_grad():
    output = backbone_lora(pixel_values=dummy_input)

print(f'\nOutput shape: {output.last_hidden_state.shape}')
print('\n✓ LoRA attachment successful!')

In [ ]:
# Extract LoRA weights
lora_weights = extract_lora_weights_peft(backbone_lora)

print(f'Number of LoRA layers: {len(lora_weights)}')
for name, weights in list(lora_weights.items())[:3]:
    print(f'  {name}:')
    print(f'    A shape: {weights["A"].shape}')
    print(f'    B shape: {weights["B"].shape}')
    print(f'    ΔW shape: {weights["delta_W"].shape}')